In [1]:
import rpy2
import numpy as np
import pandas as pd

In [2]:
%load_ext rpy2.ipython

In [3]:
%%R
print("R is working")

[1] "R is working"


In [4]:
%%R
library(rugarch)
library(rmgarch)
print("Packages loaded")

[1] "Packages loaded"


Loading required package: parallel

Attaching package: ‘rugarch’

The following object is masked from ‘package:stats’:

    sigma



In [5]:
n = 8
tril_idx = np.tril_indices(n)

def vech_to_sym(row):
    M = np.zeros((n, n))
    M[tril_idx] = row
    return M + M.T - np.diag(np.diag(M))

def load_rc(filename):
    df = pd.read_csv(filename, sep='\t', header=None)
    df.columns = range(df.shape[1])
    return np.stack([vech_to_sym(df.iloc[t].values) for t in range(len(df))])

daily_returns = pd.read_csv("data/daily_returns.txt", sep='\t', index_col=0, parse_dates=True)
returns = daily_returns

rc_full = load_rc("data/rc_kernel.txt")
rc_full = rc_full[1:]  # drop first row to match returns

realized_vars = pd.DataFrame(
    np.stack([np.diag(rc_full[t]) for t in range(len(rc_full))]),
    index=daily_returns.index,
    columns=daily_returns.columns
)

%R -i returns
%R -i realized_vars


In [6]:
%%R
library(rugarch)
library(rmgarch)

T_total = nrow(returns)
split   = T_total - 600
train = returns[1:split, ]
test  = returns[(split+1):T_total, ]

uspec = ugarchspec(mean.model = list(armaOrder = c(0,0)),
                   variance.model = list(garchOrder = c(1,1)),
                   distribution.model = "norm")
mspec = multispec(replicate(ncol(returns), uspec))
dccspec = dccspec(uspec = mspec,
                  dccOrder = c(1,1),
                  distribution = "mvnorm")

dccfit = dccfit(dccspec, data = returns, out.sample = nrow(test))
dccfor = dccforecast(dccfit, n.ahead = 1, n.roll = nrow(test) - 1)
cov_list = rcov(dccfor)

N = ncol(returns)
T = length(cov_list)

cov_array = array(NA, dim = c(N, N, T))

for (i in 1:T) {
    cov_array[,,i] = cov_list[[i]][,,1]
}

In [7]:
cov_array = %R cov_array
cov_array = cov_array.transpose(2, 0, 1)

In [8]:
cov_array[0]

array([[1.62396976e-04, 1.84917659e-04, 1.35407220e-04, 1.48814996e-04,
        1.48527328e-04, 5.90284609e-05, 7.87216201e-05, 1.80555660e-04],
       [1.84917659e-04, 5.18102862e-04, 1.78013211e-04, 1.88313463e-04,
        1.68249619e-04, 5.20104032e-05, 1.00637183e-04, 2.15917462e-04],
       [1.35407220e-04, 1.78013211e-04, 1.69092515e-04, 1.46309532e-04,
        1.57386884e-04, 5.51804474e-05, 6.15960934e-05, 1.92723465e-04],
       [1.48814996e-04, 1.88313463e-04, 1.46309532e-04, 1.76980068e-04,
        1.72693862e-04, 6.99134288e-05, 8.73870416e-05, 2.07066696e-04],
       [1.48527328e-04, 1.68249619e-04, 1.57386884e-04, 1.72693862e-04,
        2.81520070e-04, 7.39006992e-05, 6.49875866e-05, 2.94274100e-04],
       [5.90284609e-05, 5.20104032e-05, 5.51804474e-05, 6.99134288e-05,
        7.39006992e-05, 7.67599090e-05, 7.35519888e-05, 8.77515969e-05],
       [7.87216201e-05, 1.00637183e-04, 6.15960934e-05, 8.73870416e-05,
        6.49875866e-05, 7.35519888e-05, 1.60358275e-04, 7.

In [9]:
for c in cov_array:
    if not np.all(np.linalg.eigvals(c) >= 0):
        print(c)

In [10]:
realized_vars = returns ** 2
realized_vars = realized_vars.values

In [11]:
%%R
library(xts)
library(rugarch)
library(rmgarch)

time_index   = seq.Date(from = as.Date("2020-01-01"), by = "days", length.out = nrow(returns))
returns_xts  = xts(returns, order.by = time_index)
realized_xts = xts(realized_vars, order.by = time_index)

T_total = nrow(returns_xts)
split   = T_total - 600
T_test  = 600

uspec_r = ugarchspec(variance.model = list(model = "realGARCH", garchOrder = c(1,1)),
  mean.model = list(armaOrder = c(0,0)),
  distribution.model = "norm"
)

mspec_r   = multispec(replicate(ncol(returns), uspec_r))
dccspec_r = dccspec(uspec = mspec_r, dccOrder = c(1,1), distribution = "mvnorm")

dccfit_r = dccfit(
  dccspec_r,
  data        = returns_xts,
  realizedVol = realized_xts,
  out.sample  = T_test
)


realized_xts_test = realized_xts[(split + 1):T_total, ]

dccfor_r = dccforecast(
  dccfit_r,
  n.ahead     = 1,
  n.roll      = T_test - 1,
)


cov_list_r = rcov(dccfor_r)
N_R = ncol(returns_xts)
T_r = length(cov_list_r)

cov_array_r = array(NA, dim = c(N_R, N_R, T_r))
for (i in 1:T_r) {
  cov_array_r[,,i] = cov_list_r[[i]][,,1]
}

Loading required package: zoo

Attaching package: ‘zoo’

The following objects are masked from ‘package:base’:

    as.Date, as.Date.numeric


Attaching package: ‘xts’

The following objects are masked from ‘package:rmgarch’:

    first, last



In [12]:
cov_array_r = %R cov_array_r
cov_array_r = cov_array_r.transpose(2, 0, 1)

In [13]:
def save_cov_array(cov_array, filename, n=8):
    tril_idx = np.tril_indices(n)
    rows = [cov_array[t][tril_idx] for t in range(len(cov_array))]
    pd.DataFrame(rows).to_csv(filename, sep='\t', index=False, header=False)
    print(f"Saved {len(rows)} rows to {filename}")

#save_cov_array(cov_array,   "data/pred_dcc_sub.txt")
save_cov_array(cov_array_r, "data/pred_dcc_real_kernel.txt")

Saved 600 rows to data/pred_dcc_real_kernel.txt
